In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from hypergraphx import Hypergraph


RESULTS_DIR = Path("results")
DATASETS = ["contact_primary", "contact-high", "email-Enron-full", "email-Eu-full"]

rows = []

for dataset in DATASETS:
    csv_path = next((RESULTS_DIR / dataset).glob("*_wide.csv"))  # one file per folder
    df = pd.read_csv(csv_path)

    for method in ["hmrw", "hyper2vec"]:
        d = df[df["method"] == method]
        if d.empty:
            continue

        best = d.loc[d["auc"].idxmax()]

        rows.append(
            {
                "dataset": dataset,
                "method": method,
                "summary": "max",
                "auc": float(best["auc"]),
                "ap": float(best["ap"]),
                "config_id": best.get("config_id", None),
            }
        )

        rows.append(
            {
                "dataset": dataset,
                "method": method,
                "summary": "mean",
                "auc": float(d["auc"].mean()),
                "ap": float(d["ap"].mean()),
                "config_id": None,
            }
        )

summary_df = pd.DataFrame(rows).sort_values(["dataset", "method", "summary"]).reset_index(drop=True)
summary_df

,dataset,method,summary,auc,ap,config_id
0,contact-high,hmrw,max,0.937728,0.949668,hmrw_4
1,contact-high,hmrw,mean,0.931561,0.941557,None
2,contact-high,hyper2vec,max,0.930837,0.942398,h2v_5
3,contact-high,hyper2vec,mean,0.926459,0.938375,None
4,contact_primary,hmrw,max,0.866529,0.887363,hmrw_11
5,contact_primary,hmrw,mean,0.862484,0.886711,None
6,contact_primary,hyper2vec,max,0.859970,0.890247,h2v_2
7,contact_primary,hyper2vec,mean,0.851194,0.883731,None
8,email-Enron-full,hmrw,max,0.926580,0.919041,hmrw_9
9,email-Enron-full,hmrw,mean,0.920400,0.916254,None


In [2]:
diff_rows = []

for dataset in sorted(summary_df["dataset"].unique()):
    d = summary_df[summary_df["dataset"] == dataset]

    for summary_kind in ["mean", "max"]:
        hmrw_auc = d[(d["method"] == "hmrw") & (d["summary"] == summary_kind)]["auc"].iloc[0]
        h2v_auc = d[(d["method"] == "hyper2vec") & (d["summary"] == summary_kind)]["auc"].iloc[0]

        diff_rows.append(
            {
                "dataset": dataset,
                "summary": summary_kind,
                "auc_diff_hmrw_minus_hyper2vec": float(hmrw_auc - h2v_auc),
            }
        )

diff_df = pd.DataFrame(diff_rows).sort_values(["dataset", "summary"]).reset_index(drop=True)
diff_df

,dataset,summary,auc_diff_hmrw_minus_hyper2vec
0,contact-high,max,0.006891
1,contact-high,mean,0.005102
2,contact_primary,max,0.006558
3,contact_primary,mean,0.011290
4,email-Enron-full,max,-0.006152
5,email-Enron-full,mean,-0.006933
6,email-Eu-full,max,0.041673
7,email-Eu-full,mean,0.043904


In [3]:
# Best (max) comparison per dataset: HMRW vs Hyper2Vec
best = summary_df[summary_df["summary"] == "max"].copy()

rows = []
for dataset in sorted(best["dataset"].unique()):
    d = best[best["dataset"] == dataset]

    hmrw = d[d["method"] == "hmrw"].iloc[0]
    h2v = d[d["method"] == "hyper2vec"].iloc[0]

    auc_diff = float(hmrw["auc"] - h2v["auc"])
    ap_diff = float(hmrw["ap"] - h2v["ap"])

    auc_pct = float((auc_diff / h2v["auc"]) * 100) if h2v["auc"] != 0 else np.nan
    ap_pct = float((ap_diff / h2v["ap"]) * 100) if h2v["ap"] != 0 else np.nan

    rows.append(
        {
            "dataset": dataset,
            "hmrw_auc_max": float(hmrw["auc"]),
            "hmrw_ap_max": float(hmrw["ap"]),
            "hmrw_config_id": hmrw.get("config_id", None),
            "hyper2vec_auc_max": float(h2v["auc"]),
            "hyper2vec_ap_max": float(h2v["ap"]),
            "hyper2vec_config_id": h2v.get("config_id", None),
            "auc_diff_hmrw_minus_hyper2vec": auc_diff,
            "ap_diff_hmrw_minus_hyper2vec": ap_diff,
            "auc_pct_change_h2v_to_hmrw": auc_pct,
            "ap_pct_change_h2v_to_hmrw": ap_pct,
        }
    )

best_compare_df = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)
best_compare_df

,dataset,hmrw_auc_max,hmrw_ap_max,hmrw_config_id,hyper2vec_auc_max,hyper2vec_ap_max,hyper2vec_config_id,auc_diff_hmrw_minus_hyper2vec,ap_diff_hmrw_minus_hyper2vec,auc_pct_change_h2v_to_hmrw,ap_pct_change_h2v_to_hmrw
0,contact-high,0.937728,0.949668,hmrw_4,0.930837,0.942398,h2v_5,0.006891,0.007269,0.740276,0.771362
1,contact_primary,0.866529,0.887363,hmrw_11,0.859970,0.890247,h2v_2,0.006558,-0.002885,0.762621,-0.324016
2,email-Enron-full,0.926580,0.919041,hmrw_9,0.932732,0.927113,h2v_5,-0.006152,-0.008072,-0.659589,-0.870707
3,email-Eu-full,0.919540,0.932050,hmrw_11,0.877867,0.898049,h2v_5,0.041673,0.034000,4.747034,3.786032


In [6]:
hypergraphs = dict()
for dataset_name in DATASETS:    
    name = dataset_name.replace("_", "-")
    edges = []
    with open(f"data/real/{name}_graph.txt") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            edges.append([int(x) for x in line.replace(",", " ").split()])
    hypergraphs[name] = Hypergraph(edges)

In [ ]:
stats_rows = []

for name, H in hypergraphs.items():
    n_nodes = len(H.get_nodes())
    n_edges = len(H.get_edges())

    deg_dist = H.degree_distribution()  # degree -> number of nodes
    if n_nodes > 0:
        mean_node_degree = float(sum(deg * cnt for deg, cnt in deg_dist.items()) / n_nodes)
    else:
        mean_node_degree = float("nan")

    edge_sizes = H.get_sizes()
    mean_hyperedge_size = float(np.mean(edge_sizes)) if len(edge_sizes) > 0 else float("nan")

    stats_rows.append(
        {
            "dataset": name,
            "n_nodes": n_nodes,
            "n_hyperedges": n_edges,
            "mean_node_degree": mean_node_degree,
            "mean_hyperedge_size": mean_hyperedge_size,
        }
    )

dataset_stats_df = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
dataset_stats_df
    

{'contact-primary': <hypergraphx.core.hypergraph.Hypergraph at 0x1292fedd0>,
 'contact-high': <hypergraphx.core.hypergraph.Hypergraph at 0x1292d5390>,
 'email-Enron-full': <hypergraphx.core.hypergraph.Hypergraph at 0x1292d4dc0>,
 'email-Eu-full': <hypergraphx.core.hypergraph.Hypergraph at 0x129472530>}